### Load paper-year info with ROS dataset

In [ ]:
import pandas as pd
import glob
import yaml
from tqdm import tqdm

In [ ]:
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=False)

In [ ]:
import os
os.chdir('../../')

In [ ]:
def regex_search_match(input_str):
    import re
    output = re.search('^(CN-.*)', input_str, re.IGNORECASE)
    return output.group(1).replace('-', '').upper() if output else None

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
ROS_patent_paper = pd.read_csv(dataset_config['path_pcs'] + '_pcs_oa.csv', usecols=['oaid', 'patent'], engine='pyarrow').rename(columns={'oaid': 'work_id', 'patent': 'patent_id'})
ROS_patent_paper

In [ ]:
# Select China pantent cited papers
ROS_patent_paper['patent_id'] = ROS_patent_paper['patent_id'].parallel_apply(regex_search_match)
ROS_patent_paper = ROS_patent_paper.dropna().rename(columns={'work_id': 'paperid'})
ROS_patent_paper

In [ ]:
# Clean the data by removing rows with missing values and duplicates.
ROS_paper_ids = ROS_patent_paper.dropna()[['paperid']].drop_duplicates()
ROS_paper_ids

In [ ]:
# Create a list of all .csv.gz files in the directory
file_pattern = os.path.join(dataset_config['path_openalex'], 'works_year_*.csv.gz')
csv_files = glob.glob(file_pattern)

# Read each file into a DataFrame and store them in a list
dataframes = []
for file in tqdm(csv_files):
    try:
        df = pd.read_csv(file, compression='gzip')
        if len(df) >= 1:
            dataframes.append(df.rename(columns={'work_id': 'paperid'}))
    except Exception as e:
        print(f"Error reading {file}: {e}")

In [ ]:
oa_data = pd.concat(dataframes, ignore_index=True)
oa_data

In [ ]:
del dataframes

In [ ]:
# Merge patent-paper data with the MAG affiliation data on the 'paperid' column.
# A left join ensures all patent-paper pairs are kept even if they lack affiliation data.
paper_year = ROS_paper_ids.merge(oa_data, on='paperid', how='inner')
paper_year

In [ ]:
print(f'ROS matched paper in OpenAlex: {len(paper_year)} / {len(ROS_paper_ids)} = { len(paper_year) / len(ROS_paper_ids) * 100:.4f}%')

In [ ]:
patent_paper_year = ROS_patent_paper.merge(paper_year, on='paperid', how='inner')
patent_paper_year

In [ ]:
print(f'ROS matched patent-paper pair in OpenAlex: {len(patent_paper_year)} / {len(ROS_patent_paper)} = { len(patent_paper_year) / len(ROS_patent_paper) * 100:.4f}%')

In [ ]:
paper_year.to_parquet(dataset_config['path_processed'] + 'CN_CN/OA1_paper_year.parquet', index=False)